# Test inference notebook\n\nThis notebook validates that inference is really running: it loads models, runs prediction on the latest scraped image, draws bounding boxes, and saves an annotated output.

In [ ]:
from pathlib import Path\nfrom datetime import datetime\nimport cv2\nimport matplotlib.pyplot as plt\nfrom ultralytics import YOLO

In [ ]:
# Project paths\nPROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()\nRAW_DIRS = [\n    PROJECT_ROOT / 'data' / 'raw' / 'webcam_cardiff_snapshots',\n    PROJECT_ROOT / 'data' / 'raw' / 'webcam_murcia_snapshots',\n    PROJECT_ROOT / 'data' / 'raw' / 'webcam_lapalma_snapshots',\n]\nMODEL_PERSON_PATH = PROJECT_ROOT / 'data' / 'final' / 'weights_model' / 'yolo11n.pt'\nMODEL_CAR_PATH = PROJECT_ROOT / 'data' / 'final' / 'weights_model' / 'best_car.pt'\nOUTPUT_DIR = PROJECT_ROOT / 'data' / 'final' / 'image_annoted'\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nprint('PROJECT_ROOT:', PROJECT_ROOT)\nprint('MODEL_PERSON_PATH exists:', MODEL_PERSON_PATH.exists())\nprint('MODEL_CAR_PATH exists:', MODEL_CAR_PATH.exists())

In [ ]:
def resolve_class_ids(model, keywords):\n    ids = []\n    for class_id, class_name in model.names.items():\n        name = str(class_name).lower().strip()\n        if any(k in name for k in keywords):\n            ids.append(int(class_id))\n    ids = sorted(set(ids))\n    return ids if ids else None\n\ndef find_latest_raw_image(raw_dirs):\n    candidates = []\n    for d in raw_dirs:\n        if d.exists():\n            for p in d.glob('*'):\n                if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'} and p.is_file():\n                    candidates.append(p)\n    if not candidates:\n        raise FileNotFoundError('No raw webcam image found in data/raw/webcam_*_snapshots')\n    return max(candidates, key=lambda p: p.stat().st_mtime)

In [ ]:
# Load models\nmodel_person = YOLO(str(MODEL_PERSON_PATH))\nmodel_car = YOLO(str(MODEL_CAR_PATH))\n\nperson_classes = resolve_class_ids(model_person, ['person', 'people', 'pedestrian'])\ncar_classes = resolve_class_ids(model_car, ['car', 'cars', 'vehicle', 'truck', 'bus'])\n\nprint('person_classes:', person_classes)\nprint('car_classes:', car_classes)

In [ ]:
# Pick latest scraped image\nimage_path = find_latest_raw_image(RAW_DIRS)\nprint('Input image:', image_path)\n\nimg = cv2.imread(str(image_path))\nif img is None:\n    raise RuntimeError(f'Cannot read image: {image_path}')\n\ndetections = []\ndetection_count = {'person': 0, 'car': 0}\n\n# Person model inference\nres_person = model_person.predict(source=str(image_path), classes=person_classes, conf=0.25, save=False, verbose=False)[0]\nif res_person.boxes is not None:\n    for box in res_person.boxes:\n        conf = round(float(box.conf[0]), 2)\n        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]\n        cv2.rectangle(img, (x1, y1), (x2, y2), (219, 142, 0), 2)\n        cv2.putText(img, f'person {conf}', (x1, max(10, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (219, 142, 0), 1)\n        detections.append({'class': 'person', 'confidence': conf})\n        detection_count['person'] += 1\n\n# Car model inference\nres_car = model_car.predict(source=str(image_path), classes=car_classes, conf=0.25, save=False, verbose=False)[0]\nif res_car.boxes is not None:\n    for box in res_car.boxes:\n        conf = round(float(box.conf[0]), 2)\n        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]\n        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 220), 2)\n        cv2.putText(img, f'car {conf}', (x1, max(10, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 220), 1)\n        detections.append({'class': 'car', 'confidence': conf})\n        detection_count['car'] += 1\n\nts = datetime.now().strftime('%Y%m%d_%H%M%S')\nout_path = OUTPUT_DIR / f'notebook_inference_{ts}.png'\nok = cv2.imwrite(str(out_path), img)\nif not ok:\n    raise RuntimeError(f'Failed to write output image: {out_path}')\n\nprint('Output image:', out_path)\nprint('Detections total:', len(detections))\nprint('Detections by class:', detection_count)

In [ ]:
# Display annotated output\nimg_rgb = cv2.cvtColor(cv2.imread(str(out_path)), cv2.COLOR_BGR2RGB)\nplt.figure(figsize=(12, 7))\nplt.imshow(img_rgb)\nplt.axis('off')\nplt.title(f'Annotated output | person={detection_count["person"]} car={detection_count["car"]}')\nplt.show()